In [ ]:
# Install onnxruntime from local wheel if available (KAGGLE_NO_INTERNET=1)
import subprocess, sys, os, glob as _glob

# Auto-detect the model dataset directory
def _find_dir(pattern):
    direct = f"/kaggle/input/{pattern}"
    if os.path.isdir(direct):
        return direct
    for root, dirs, files in os.walk("/kaggle/input"):
        if pattern in root:
            return root
    return direct

WHEEL_DIR = _find_dir("birdclef-perch-models")

# First, check if onnxruntime is already installed
try:
    import onnxruntime
    print(f"onnxruntime already installed: {onnxruntime.__version__}")
except ImportError:
    print(f"Dataset dir: {WHEEL_DIR}")
    print(f"Contents: {sorted(os.listdir(WHEEL_DIR))}")

    # Find any .whl file
    wheels = sorted(_glob.glob(os.path.join(WHEEL_DIR, "*.whl")))
    if wheels:
        wheel_path = wheels[0]
        print(f"\nInstalling onnxruntime from {os.path.basename(wheel_path)}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--no-deps", wheel_path],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        import onnxruntime
        print(f"onnxruntime {onnxruntime.__version__} OK")
    else:
        raise RuntimeError(
            f"No .whl file found in {WHEEL_DIR}. "
            "Make sure dataset v2+ with onnxruntime wheel is attached."
        )

BirdCLEF 2026 -- Test Submission: Perch ONNX baseline
=========================================================
Minimal submission using Perch ONNX logits directly,
mapped to BirdCLEF 234 species. No trained classifier needed.

**Dataset requis**: hellodave2035/birdclef-perch-models
**Runtime cible**: < 60 min CPU


In [ ]:
import os
os.environ["KAGGLE_NO_INTERNET"] = "1"

import numpy as np
import pandas as pd
import onnxruntime as ort
import soundfile as sf
import gc
import time
from pathlib import Path


## 1. Configuration


In [ ]:
# Auto-detect Kaggle input paths (handles both /kaggle/input/<slug> and /kaggle/input/datasets/<user>/<slug>)
import os, glob as _glob

def find_input_dir(pattern):
    """Find a Kaggle input directory matching the pattern."""
    # Try direct slug first
    direct = f"/kaggle/input/{pattern}"
    if os.path.isdir(direct):
        return direct
    # Try with datasets/<user>/ prefix
    for user_dir in os.listdir("/kaggle/input"):
        user_path = f"/kaggle/input/{user_dir}"
        if os.path.isdir(user_path) and user_dir not in ("birdclef-2026", pattern):
            candidate = f"{user_path}/{pattern}"
            if os.path.isdir(candidate):
                return candidate
    # Recursive search
    for root, dirs, files in os.walk("/kaggle/input"):
        if pattern in root:
            return root
    return direct  # fallback

MODEL_DIR = find_input_dir("birdclef-perch-models")
COMP_DIR  = find_input_dir("birdclef-2026")

print(f"COMP_DIR:  {COMP_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"COMP_DIR contents: {sorted(os.listdir(COMP_DIR))[:10]}...")
print(f"MODEL_DIR contents: {sorted(os.listdir(MODEL_DIR))}")

# Derived paths
TEST_DIR  = f"{COMP_DIR}/test_soundscapes"
SAMPLE_SUB_PATH = f"{COMP_DIR}/sample_submission.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"

SR = 32000
DURATION = 5
SEGMENT_SAMPLES = SR * DURATION  # 160000
STEP_SEC = 2.5
STEP_SAMPLES = int(SR * STEP_SEC)
BATCH_SIZE = 32

# ONNX runtime
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opts.intra_op_num_threads = 8
providers = ["CPUExecutionProvider"]

## 2. Load Models & Data


In [ ]:
print("=" * 50)
print("Loading Perch ONNX and metadata...")
print("=" * 50)

# Perch ONNX
perch_path = f"{MODEL_DIR}/perch_v2.onnx"
perch_sess = ort.InferenceSession(perch_path, sess_opts, providers=providers)
perch_in = perch_sess.get_inputs()[0].name
perch_out = [o.name for o in perch_sess.get_outputs()]
print(f"Perch loaded. Output keys: {perch_out}")

# Perch labels -> BirdCLEF mapping
perch_labels_df = pd.read_csv(f"{MODEL_DIR}/labels.csv")
print(f"Perch classes: {len(perch_labels_df)}")

# BirdCLEF species from sample submission
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_COLS = [c for c in sample_sub.columns if c != "row_id"]
N_SPECIES = len(SPECIES_COLS)
species_to_idx = {sp: i for i, sp in enumerate(SPECIES_COLS)}
print(f"BirdCLEF species to predict: {N_SPECIES}")


## 3. Build Perch -> BirdCLEF Mapping


In [ ]:
perch_labels_list = []
for i, row in perch_labels_df.iterrows():
    label = str(row.get("label", row.get("scientific_name", ""))).lower().strip()
    perch_labels_list.append(label)

perch_to_bc = {}  # perch_idx -> [bc_idx, ...]
bc_to_perch = {}  # bc_idx -> perch_idx

for bc_sp in SPECIES_COLS:
    bc_sp_lower = bc_sp.lower().replace("_", " ").strip()
    matched = False
    # Try exact match first
    for pi, pl in enumerate(perch_labels_list):
        if pl == bc_sp_lower:
            bc_to_perch[species_to_idx[bc_sp]] = pi
            perch_to_bc.setdefault(pi, []).append(species_to_idx[bc_sp])
            matched = True
            break
    # Try genus match
    if not matched:
        genus = bc_sp_lower.split()[0] if " " in bc_sp_lower else bc_sp_lower
        for pi, pl in enumerate(perch_labels_list):
            if pl.startswith(genus + " "):
                bc_to_perch[species_to_idx[bc_sp]] = pi
                perch_to_bc.setdefault(pi, []).append(species_to_idx[bc_sp])
                matched = True
                break

matched = len(bc_to_perch)
print(f"Mapping: {matched}/{N_SPECIES} species matched to Perch classes")

unmatched = [sp for sp in SPECIES_COLS if species_to_idx[sp] not in bc_to_perch]
if unmatched:
    print(f"Unmatched ({len(unmatched)}): {unmatched[:15]}...")
else:
    print("All species matched!")


## 4. Inference Functions


In [ ]:
def load_audio_segments(file_path):
    """Load soundscape -> array of 5s sliding-window segments (N, 160000)."""
    y, file_sr = sf.read(file_path, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if file_sr != SR:
        import librosa
        y = librosa.resample(y, orig_sr=file_sr, target_sr=SR)

    total = len(y)
    if total < SEGMENT_SAMPLES:
        y = np.pad(y, (0, SEGMENT_SAMPLES - total))
        total = len(y)

    segments = []
    for start in range(0, total - SEGMENT_SAMPLES + 1, STEP_SAMPLES):
        segments.append(y[start:start + SEGMENT_SAMPLES])

    if not segments:
        segments.append(y[:SEGMENT_SAMPLES])

    return np.stack(segments).astype(np.float32)


def predict_soundscape(file_path):
    """Perch inference -> BirdCLEF probabilities for all segments."""
    waveforms = load_audio_segments(file_path)  # (N, 160000)
    n_seg = len(waveforms)

    all_probs = np.zeros((n_seg, N_SPECIES), dtype=np.float32)

    for i in range(0, n_seg, BATCH_SIZE):
        batch_wav = waveforms[i:i + BATCH_SIZE]
        feed = {perch_in: batch_wav}
        outs = perch_sess.run(perch_out, feed)
        out_dict = dict(zip(perch_out, outs))

        # Get logits
        logits = None
        for key in ["label", "logits"]:
            if key in out_dict:
                logits = out_dict[key]
                break
        if logits is None:
            continue

        # Apply sigmoid: logits -> probabilities
        probs = 1.0 / (1.0 + np.exp(-logits))  # (B, ~15000)

        # Map Perch classes to BirdCLEF species
        bc_probs = np.zeros((len(batch_wav), N_SPECIES), dtype=np.float32)
        for bc_idx, perch_idx in bc_to_perch.items():
            if perch_idx < probs.shape[1]:
                bc_probs[:, bc_idx] = probs[:, perch_idx]

        all_probs[i:i + len(batch_wav)] = bc_probs

    return all_probs


## 5. Run Inference on Test Soundscapes


In [ ]:
print("\n" + "=" * 50)
print("Running inference on test soundscapes...")
print("=" * 50)

test_files = sorted([
    f for f in os.listdir(TEST_DIR)
    if f.endswith((".ogg", ".mp3"))
])
print(f"Test soundscapes: {len(test_files)}")

t_start = time.time()
results = []

for idx, file in enumerate(test_files):
    file_path = os.path.join(TEST_DIR, file)
    soundscape_id = os.path.splitext(file)[0]

    try:
        probs = predict_soundscape(file_path)
    except Exception as e:
        print(f"  [ERROR] {file}: {e}")
        # Fill with zeros for expected rows
        expected = sample_sub[
            sample_sub["row_id"].str.startswith(soundscape_id)
        ]
        probs = np.zeros((len(expected), N_SPECIES), dtype=np.float32)

    n_seg = probs.shape[0]
    for seg_idx in range(n_seg):
        end_time = int((seg_idx + 1) * STEP_SEC)
        row_id = f"{soundscape_id}_{end_time}"
        results.append([row_id] + probs[seg_idx].tolist())

    # Progress every 10 files
    if (idx + 1) % 10 == 0 or idx == 0:
        elapsed = time.time() - t_start
        eta = elapsed / (idx + 1) * len(test_files)
        print(f"  [{idx+1:3d}/{len(test_files)}] {file} "
              f"({n_seg} segments) - {elapsed:.0f}s / ~{eta:.0f}s")
        gc.collect()

total_infer_time = time.time() - t_start
print(f"\nInference done: {total_infer_time:.0f}s "
      f"({total_infer_time/60:.1f} min)")


## 6. Build Submission CSV


In [ ]:
print(f"\nBuilding submission DataFrame ({len(results)} rows)...")
sub = pd.DataFrame(results, columns=["row_id"] + SPECIES_COLS)

# Align with sample_submission
sub = sample_sub[["row_id"]].merge(sub, on="row_id", how="left").fillna(0.0)
sub = sub[sample_sub.columns]

# Validate
assert sub.shape == sample_sub.shape, \
    f"Shape mismatch: {sub.shape} vs {sample_sub.shape}"
assert list(sub.columns) == list(sample_sub.columns), "Column mismatch!"
assert sub.isnull().sum().sum() == 0, "NaN values detected!"

# Save
sub.to_csv(OUTPUT_PATH, index=False)
print(f"[OK] Saved to {OUTPUT_PATH}")
print(f"     Rows: {len(sub)}, Columns: {len(sub.columns)}")

# Quick stats
numeric = sub[SPECIES_COLS]
print(f"     Mean prob:  {numeric.values.mean():.6f}")
print(f"     Max prob:   {numeric.values.max():.6f}")
active = (numeric.max() > 0.01).sum()
print(f"     Active species (>0.01): {active}/{N_SPECIES}")


## 7. Done!


In [ ]:
print(f"\n{'=' * 50}")
print("SUBMISSION READY")
print(f"{'=' * 50}")
print(f"File: {OUTPUT_PATH}")
print(f"Total time: {total_infer_time:.0f}s ({total_infer_time/60:.1f} min)")
print("\nSubmit this file on the competition page!")
